# Nível 1 — Tratamento de dados e primeira análise com LLM

Triagem de Prevenção à Lavagem de Dinheiro sobre 20 operações de 6 clientes.

**Princípio que organiza o notebook inteiro:** cálculo é pandas, interpretação é LLM.
Soma, mediana, contagem e comparação com limite acontecem na Parte A. A LLM da Parte B
recebe os números já prontos e só redige o parecer — se ela precisasse somar algo para
responder, o insumo estaria incompleto.

A lógica de tratamento e as regras vivem em `pipeline.py`, importado aqui. Foi escrito
como módulo desde o começo justamente porque o Nível 2 roda o mesmo tratamento sobre uma
base 16x maior — lá, muda só o caminho do arquivo.

In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import pipeline as p

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

CAMINHO = "../dados/dados_nivel_1.json"
bruto, taxa = p.carregar(CAMINHO)

print(f"Operações carregadas: {len(bruto)}")
print(f"Taxa de câmbio do arquivo: USD 1,00 = BRL {taxa}")
bruto.head()

Operações carregadas: 20
Taxa de câmbio do arquivo: USD 1,00 = BRL 5.4


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


## 1. Diagnóstico da base crua

Antes de tratar, olhar. Esta célula não altera nada — só mede o estrado.

In [2]:
diag = p.diagnosticar(bruto)
print(diag.resumo())

Linhas: 20 | Clientes: 6
Duplicatas exatas (todos os campos iguais): 1
IDs repetidos: 1 -> ['OP-0007']
  ...com conteúdo divergente: nenhum
Operações com data nula: 1
Moedas presentes: {'BRL': 19, 'USD': 1}
Valores <= 0: 0
Datas fora do padrão ISO: nenhuma


In [3]:
# As linhas problemáticas, nomeadas
print("── Linhas duplicadas ──")
display(bruto[bruto.duplicated(keep=False)].sort_values("id"))

print("── Linhas sem data ──")
display(bruto[bruto["data"].isna()])

print("── Linhas em moeda estrangeira ──")
display(bruto[bruto["moeda"] != "BRL"])

── Linhas duplicadas ──


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


── Linhas sem data ──


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


── Linhas em moeda estrangeira ──


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional


## 2. Achados e decisões de tratamento

Três problemas de qualidade na base, todos com origem plausível em captura de sistema legado.

---

### 2.1 Duplicata exata — `OP-0007` (CLI-A-3)

**O que é.** A operação aparece duas vezes com *todos* os campos idênticos, inclusive o ID.
Verifiquei que não há ID repetido com conteúdo divergente — o que muda o tratamento: conteúdo
idêntico é reenvio/reprocessamento do legado e pode ser removido com segurança. Se houvesse
divergência, seria conflito de integridade e eu não teria como escolher sozinho qual linha
vale — viraria exceção para tratamento manual, não decisão de código.

**Tratamento.** `drop_duplicates()` sobre todos os campos. 20 → 19 linhas.

**Por que importa (e não é detalhe de higiene).** Sem deduplicar, CLI-A-3 passa a ter 4 operações
em 2026-03-05 somando R$ 65.700 e **dispara a Regra 1 de fracionamento**. Depois de remover a
duplicata: 3 operações, R$ 48.500, não dispara. Ou seja — a falha do sistema legado fabricava um
alerta de PLD do nada. A prova numérica está na seção 6.

---

### 2.2 Data ausente — `OP-0017` (CLI-A-5)

**O que é.** `data: null`, com a observação do próprio sistema: *"data nao capturada pelo sistema"*.
O campo `valor` está preenchido e é consistente, então é falha de captura de um atributo, não
registro corrompido.

**Tratamento.** **Mantenho a linha** e marco com `sem_data=True`. A Regra 1 (que depende de data)
ignora essas linhas explicitamente; a Regra 2 e as agregações de volume continuam contando com elas.

**Alternativas que descartei, e por quê:**

| Opção | Por que não |
|---|---|
| Descartar a linha | Remove R$ 4.300 do volume do cliente e tira uma operação do denominador da Regra 2. Em PLD, perder operação por falha de captura **subnotifica risco** — o erro mais caro dos dois |
| Imputar uma data (mediana, vizinha, primeiro do mês) | Fabrica evidência. Se a data imputada colidir com outras operações do cliente, eu **crio** um alerta de fracionamento que não existe. Inventar cronologia em contexto regulatório é indefensável |

Manter e excluir seletivamente preserva o volume real sem inventar cronologia. Num sistema de
produção, isso também deveria disparar um alerta de qualidade para o time de origem — o dado
está errado na fonte, não aqui.

---

### 2.3 Moeda estrangeira — `OP-0013` (CLI-A-4)

**O que é.** 12.000 **USD**, observação *"remessa internacional"*. É a única operação não-BRL da base.

**Tratamento.** Converto pela taxa que vem dentro do arquivo (5,4), criando a coluna `valor_brl`.
12.000 USD → **R$ 64.800**. Todas as regras e agregações usam `valor_brl`, nunca `valor`.

**Por que importa.** Sem converter, essa operação vale "12.000" na comparação e passa despercebida.
Convertida, ela é **a maior operação da base inteira** e é o único caso que a Regra 2 captura no
Nível 1. Sem essa etapa, a Regra 2 não sinaliza absolutamente nada.

---

### 2.4 O que verifiquei e estava limpo

Registro para deixar claro que a ausência de tratamento foi decisão, não descuido:

- `canal`, `tipo` e `moeda` sem variação de grafia ou caixa — nada a normalizar
- Todos os valores positivos, nenhum zero
- Todas as datas preenchidas estão em ISO `YYYY-MM-DD` válido
- Nenhum `cliente_id` malformado

**Não inventei limpeza onde não havia problema.** Normalizar campos já normalizados adiciona código
sem adicionar controle.

---

### 2.5 Ordem das operações — não é arbitrária

`deduplicar → converter → datar`, nessa ordem:

1. **Deduplicar primeiro.** Se a mediana for calculada antes, a linha repetida entra duas vezes na
   distribuição e desloca o resultado da Regra 2. (Na base do Nível 2 isso muda o status de um
   cliente real.)
2. **Converter depois.** Comparar valor com limite em moedas diferentes é comparar coisas diferentes.
3. **Datar por último**, com `errors="coerce"`, para que a data ausente vire `NaT` de forma explícita
   em vez de quebrar o parse.

In [4]:
df = p.limpar(bruto, taxa)

print(f"{len(bruto)} linhas brutas → {len(df)} após tratamento")
print(f"Operações sem data mantidas: {df['sem_data'].sum()}")
print(f"Operações convertidas de USD: {(df['moeda'] != 'BRL').sum()}")

df[["id", "cliente_id", "data", "valor", "moeda", "valor_brl", "sem_data"]]

20 linhas brutas → 19 após tratamento
Operações sem data mantidas: 1
Operações convertidas de USD: 1


,id,cliente_id,data,valor,moeda,valor_brl,sem_data
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,"18,100.00",False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,"17,300.00",False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,"18,800.00",False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,"3,300.00",False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,"25,900.00",False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,"27,000.00",False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,"17,200.00",False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,"15,200.00",False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,"16,100.00",False
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,"3,800.00",False


## 3. Agregações

**Decisão sobre "volume transacionado":** soma **absoluta**, sem sinal por direção do fluxo.
Em PLD interessa o giro (quanto passou pela conta), não o saldo líquido. Entrar R$ 100 mil e
sair R$ 100 mil é justamente o padrão que se quer enxergar — no líquido, esse caso zeraria.

In [5]:
volume = p.volume_por_cliente(df)
print("Volume total transacionado por cliente (BRL)")
volume

Volume total transacionado por cliente (BRL)


,volume_brl,n_operacoes,ticket_mediano
cliente_id,,,
CLI-A-4,"79,500.00",4,"5,450.00"
CLI-A-1,"57,500.00",4,"17,700.00"
CLI-A-2,"52,900.00",2,"26,450.00"
CLI-A-3,"48,500.00",3,"16,100.00"
CLI-A-5,"16,900.00",4,"3,600.00"
CLI-A-6,"10,200.00",2,"5,100.00"


In [6]:
canais = p.operacoes_por_canal(df)
print("Quantidade de operações por canal")
canais

Quantidade de operações por canal


,n_operacoes,volume_brl,pct_operacoes
canal,,,
pix,8,"101,400.00",42.10
ted,5,"143,500.00",26.30
boleto,3,"11,100.00",15.80
cartao,2,"5,200.00",10.50
especie,1,"4,300.00",5.30


## 4. Regras determinísticas

Antes do código, as **interpretações de fronteira**. O enunciado está em linguagem natural e
cada verbo tem uma leitura — registro a minha para que a regra seja auditável:

### Regra 1 — Fracionamento (sinaliza o **cliente**, no par cliente+data)

| Trecho do enunciado | Implementação | Por quê |
|---|---|---|
| "3 ou mais operações" | `n >= 3` | inclusivo, literal |
| "soma **ultrapassa** R$ 50.000" | `soma > 50_000` | *ultrapassar* é exceder; exatamente 50.000 não ultrapassa |
| "nenhuma operação isolada **atinge** R$ 20.000" | `max < 20_000` | *atingir* é alcançar; exatamente 20.000 atinge, logo desqualifica |
| — | soma sobre `valor_brl` | comparar com limite em BRL exige valor em BRL |
| — | linhas sem data ficam fora | não existe "mesma data" para elas |

### Regra 2 — Valor atípico (sinaliza a **operação**)

| Trecho | Implementação | Por quê |
|---|---|---|
| "superior a 5× a mediana" | `valor_brl > 5 * mediana` | *superior a* é estrito |
| "mediana daquele mesmo cliente" | mediana sobre `valor_brl`, pós-limpeza | a mediana inclui a própria operação testada — leitura mais direta e mais **conservadora**, já que o outlier puxa a mediana para cima e a regra dispara menos |
| "clientes com 4 ou mais operações" | `count >= 4`, contagem **após** a limpeza | uma duplicata podia empurrar um cliente de 3 para 4 operações e fazê-lo entrar na regra sem motivo |
| — | linhas sem data **permanecem** | a Regra 2 não é temporal; excluí-las mudaria mediana e contagem sem justificativa |

Os limiares estão nomeados no topo de `pipeline.py`, não espalhados na lógica: limiar de compliance
é **política**, não constante de código — num sistema real vem de configuração e muda sem deploy.

In [7]:
df = p.aplicar_regras(df)

print("Flags adicionadas ao DataFrame:")
df[["id", "cliente_id", "data", "valor_brl", "flag_fracionamento", "flag_valor_atipico"]]

Flags adicionadas ao DataFrame:


,id,cliente_id,data,valor_brl,flag_fracionamento,flag_valor_atipico
0,OP-0001,CLI-A-1,2026-03-09,"18,100.00",True,False
1,OP-0002,CLI-A-1,2026-03-09,"17,300.00",True,False
2,OP-0003,CLI-A-1,2026-03-09,"18,800.00",True,False
3,OP-0004,CLI-A-1,2026-03-21,"3,300.00",False,False
4,OP-0005,CLI-A-2,2026-03-14,"25,900.00",False,False
5,OP-0006,CLI-A-2,2026-03-14,"27,000.00",False,False
6,OP-0007,CLI-A-3,2026-03-05,"17,200.00",False,False
7,OP-0008,CLI-A-3,2026-03-05,"15,200.00",False,False
8,OP-0009,CLI-A-3,2026-03-05,"16,100.00",False,False
9,OP-0010,CLI-A-4,2026-03-03,"3,800.00",False,False


In [8]:
print("Operações sinalizadas")
display(df[df["flag_qualquer"]][
    ["id", "cliente_id", "data", "valor_brl", "canal", "tipo",
     "flag_fracionamento", "flag_valor_atipico"]
])

print(f"\nClientes com fracionamento : {sorted(df.loc[df['flag_fracionamento'],'cliente_id'].unique())}")
print(f"Clientes com valor atípico: {sorted(df.loc[df['flag_valor_atipico'],'cliente_id'].unique())}")

Operações sinalizadas


,id,cliente_id,data,valor_brl,canal,tipo,flag_fracionamento,flag_valor_atipico
0,OP-0001,CLI-A-1,2026-03-09,"18,100.00",pix,transferencia_enviada,True,False
1,OP-0002,CLI-A-1,2026-03-09,"17,300.00",pix,transferencia_enviada,True,False
2,OP-0003,CLI-A-1,2026-03-09,"18,800.00",ted,transferencia_enviada,True,False
12,OP-0013,CLI-A-4,2026-03-24,"64,800.00",ted,transferencia_recebida,False,True



Clientes com fracionamento : ['CLI-A-1']
Clientes com valor atípico: ['CLI-A-4']


## 5. Validação — a Regra 1 acerta o alvo e recusa os vizinhos

O enunciado pede prova de que a regra captura o caso certo **e não** captura um caso parecido.
A base oferece dois quase-positivos, e eles falham por motivos diferentes — por isso a tabela
abaixo mostra **cada condição isolada**, e não só o booleano final. Saber *qual* condição barrou
cada caso é o que torna a regra auditável.

In [9]:
grupos = p.grupos_cliente_data(df)
grupos[grupos["n_operacoes"] >= 2]

,,n_operacoes,soma_brl,maior_operacao,cond_3_ou_mais_ops,cond_soma_acima_50k,cond_nenhuma_atinge_20k,fracionamento
cliente_id,data_dt,,,,,,,
CLI-A-1,2026-03-09,3,"54,200.00","18,800.00",True,True,True,True
CLI-A-2,2026-03-14,2,"52,900.00","27,000.00",False,True,False,False
CLI-A-3,2026-03-05,3,"48,500.00","17,200.00",True,False,True,False


**Leitura dos três casos:**

| Cliente | Data | n | Soma | Maior op. | Veredito |
|---|---|---|---|---|---|
| **CLI-A-1** | 09/03 | 3 | R$ 54.200 | R$ 18.800 | ✅ **CAPTURA** |
| **CLI-A-2** | 14/03 | 2 | R$ 52.900 | R$ 27.000 | ❌ não captura |
| **CLI-A-3** | 05/03 | 3 | R$ 48.500 | R$ 17.200 | ❌ não captura |

**CLI-A-1 — captura, e deve capturar.** Três operações no mesmo dia, R$ 54.200 no total, nenhuma
chegando a R$ 20.000. Duas para a mesma contraparte via PIX, uma via TED para outra. É o desenho
clássico de fracionamento: o valor foi partido em pedaços que individualmente não chamam atenção.

**CLI-A-2 — não captura, e é o contraexemplo pedido.** Movimentou R$ 52.900 no mesmo dia, *acima*
do limite de R$ 50.000 — a condição de soma passa. Mas foram **2** operações, de R$ 25.900 e
R$ 27.000. Falha em duas condições ao mesmo tempo. E falha com razão: quem está fracionando para
escapar de um limite não faz transferência de R$ 27.000. Aqui há volume alto, que é outro tipo de
risco, mas **não é fracionamento**. Uma regra que capturasse esse caso estaria confundindo dois
fenômenos distintos.

**CLI-A-3 — não captura, mas só porque os dados foram tratados.** Este é o caso mais interessante,
e é o que amarra a Parte A inteira: veja a próxima célula.

In [10]:
# Prova de que a etapa de limpeza é parte do controle, não preparação dela
def variante(dedup=True, converte=True, drop_sem_data=False):
    b, tx = p.carregar(CAMINHO)
    d = b.drop_duplicates().copy() if dedup else b.copy()
    d["valor_brl"] = d["valor"].where(d["moeda"] != "USD", d["valor"] * (tx if converte else 1))
    d["data_dt"] = pd.to_datetime(d["data"], errors="coerce")
    d["sem_data"] = d["data_dt"].isna()
    if drop_sem_data:
        d = d[~d["sem_data"]].reset_index(drop=True)
    d = p.aplicar_regras(d)
    return (sorted(d.loc[d["flag_fracionamento"], "cliente_id"].unique()),
            sorted(d.loc[d["flag_valor_atipico"], "cliente_id"].unique()))

cenarios = {
    "pipeline completo":      variante(),
    "SEM deduplicar":         variante(dedup=False),
    "SEM converter USD":      variante(converte=False),
    "DESCARTANDO linhas sem data": variante(drop_sem_data=True),
}

pd.DataFrame(
    [{"cenário": k,
      "Regra 1 sinaliza": ", ".join(v[0]) or "—",
      "Regra 2 sinaliza": ", ".join(v[1]) or "—"} for k, v in cenarios.items()]
).set_index("cenário")

,Regra 1 sinaliza,Regra 2 sinaliza
cenário,,
pipeline completo,CLI-A-1,CLI-A-4
SEM deduplicar,"CLI-A-1, CLI-A-3",CLI-A-4
SEM converter USD,CLI-A-1,—
DESCARTANDO linhas sem data,CLI-A-1,CLI-A-4


### O que essa tabela demonstra

**Sem deduplicar, CLI-A-3 é sinalizado por fracionamento.** As 4 linhas (uma delas fantasma) somam
R$ 65.700 e nenhuma atinge R$ 20.000 — a regra dispara. Um analista humano abriria o caso, gastaria
tempo, e descobriria que a terceira operação nunca existiu. **A duplicata do sistema legado produzia
um falso positivo de PLD.**

**Sem converter USD, a Regra 2 não sinaliza nada.** A remessa de 12.000 USD é, em BRL, a maior
operação da base — R$ 64.800 contra uma mediana de R$ 5.450 no cliente. Sem conversão ela vale
"12.000" na comparação e some. A regra continuaria rodando, sem erro, retornando vazio.

**Conclusão que atravessa o notebook:** a limpeza não é etapa preparatória para a análise — ela **é**
parte do controle. Uma regra correta sobre dados sujos produz alerta errado nas duas direções: cria
alarme onde não há risco (CLI-A-3) e silencia risco onde há (CLI-A-4). Nenhuma das duas falhas
apareceria como erro de execução.

In [11]:
# Regra 2 — a aritmética exposta, cliente a cliente
detalhe = df.groupby("cliente_id")["valor_brl"].agg(
    n_operacoes="size", mediana="median", maior_operacao="max")
detalhe["limite_5x"] = detalhe["mediana"] * p.FATOR_VALOR_ATIPICO
detalhe["elegivel_4_ops"] = detalhe["n_operacoes"] >= p.MIN_OPS_VALOR_ATIPICO
detalhe["dispara"] = detalhe["elegivel_4_ops"] & (detalhe["maior_operacao"] > detalhe["limite_5x"])
detalhe.round(2)

,n_operacoes,mediana,maior_operacao,limite_5x,elegivel_4_ops,dispara
cliente_id,,,,,,
CLI-A-1,4,"17,700.00","18,800.00","88,500.00",True,False
CLI-A-2,2,"26,450.00","27,000.00","132,250.00",False,False
CLI-A-3,3,"16,100.00","17,200.00","80,500.00",False,False
CLI-A-4,4,"5,450.00","64,800.00","27,250.00",True,True
CLI-A-5,4,"3,600.00","7,000.00","18,000.00",True,False
CLI-A-6,2,"5,100.00","8,800.00","25,500.00",False,False


**CLI-A-4 é o único caso.** Mediana de R$ 5.450, limite de R$ 27.250, e uma operação de R$ 64.800 —
quase 12× a mediana. As três outras operações do cliente ficam entre R$ 3.800 e R$ 5.800; a quarta
destoa por uma ordem de grandeza.

**CLI-A-2, CLI-A-3 e CLI-A-6 são excluídos por terem menos de 4 operações** — e o corte é acertado:
com 2 ou 3 pontos a mediana não descreve padrão nenhum, e qualquer valor um pouco maior estouraria
5×. A regra estaria medindo ruído amostral, não anomalia.

**Limitação que já registro:** mesmo com 4 operações a mediana é frágil. CLI-A-5 tem mediana de
R$ 3.600 e limite de R$ 18.000 — bastaria uma operação corriqueira de R$ 20 mil para disparar.
Na base do Nível 2, com clientes de 10+ operações, esse efeito produz falsos positivos em série.
Volto a isso no confronto.

In [12]:
# Snapshot final da Parte A — insumo que a Parte B vai interpretar
resumo = p.ranking_clientes(df, top=None)
print("Panorama consolidado por cliente")
resumo

Panorama consolidado por cliente


,flags_regra1,flags_regra2,volume_brl,n_operacoes,total_flags
cliente_id,,,,,
CLI-A-4,0,1,"79,500.00",4,1
CLI-A-1,1,0,"57,500.00",4,1
CLI-A-2,0,0,"52,900.00",2,0
CLI-A-3,0,0,"48,500.00",3,0
CLI-A-5,0,0,"16,900.00",4,0
CLI-A-6,0,0,"10,200.00",2,0


---

# Parte B — Análise com LLM

Daqui em diante a LLM entra, e entra num papel **estritamente delimitado**: interpretar e redigir.
Todos os números que ela vai citar já foram calculados acima, em pandas. Ela não soma, não compara
com limite e não decide se algo ultrapassou threshold — recebe o resultado pronto e escreve o parecer.

**Cliente escolhido: CLI-A-4.** É o único sinalizado pela Regra 2, e é o caso mais rico para
interpretação: a operação atípica é uma remessa internacional recebida, em conta cujas outras três
operações são pagamentos de R$ 3-6 mil. Há uma história para ler ali que a regra sozinha não conta.

A camada de acesso está em `llm.py`, com cache em disco, retry realimentado e telemetria por chamada.

In [ ]:
import llm

CLIENTE_ALVO = "CLI-A-4"
dossie = p.dossie_cliente(df, CLIENTE_ALVO)

print("Dossiê enviado à LLM — tudo pré-calculado, nada para ela somar:")
print(json.dumps(dossie, indent=2, ensure_ascii=False))

## 6. O contrato de saída

```python
class ParecerPLD(BaseModel):
    nivel_risco: Literal["baixo", "medio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str
```

`Literal` no nível de risco é a decisão mais importante do schema. Se o modelo responder
`"altíssimo"`, `"ALTO"` ou `"muito alto"`, a validação **rejeita** em vez de aceitar um valor que
quebraria a comparação automática do Nível 2. Vocabulário fechado é o que torna a saída utilizável
a jusante — sem isso, o confronto entre regra e modelo viraria *string matching* frouxo.

In [ ]:
from IPython.display import Markdown

print("Schema exigido:")
print(json.dumps(llm.ParecerPLD.model_json_schema(), indent=2, ensure_ascii=False))

## 7. Prompt v1 — mínimo

Só o dossiê e o formato pedido. Serve de linha de base: quero ver o que o modelo faz sem
enquadramento nenhum.

In [ ]:
SYSTEM_V1 = "Voce analisa operacoes financeiras. Responda apenas com um objeto JSON."

USER_V1 = f"""Analise o cliente abaixo e produza um parecer de risco.

{json.dumps(dossie, ensure_ascii=False, indent=2)}

Responda em JSON com os campos: nivel_risco, tipologia_suspeita, red_flags, justificativa."""

parecer_v1, tel_v1 = llm.chamar(SYSTEM_V1, USER_V1)

print(f"status={tel_v1.status} | tentativas={tel_v1.tentativas} | cacheado={tel_v1.cacheado}")
print(f"tokens: {tel_v1.tokens_entrada} entrada + {tel_v1.tokens_saida} saida")
print(f"latencia: {tel_v1.latencia_s}s | custo estimado: US$ {tel_v1.custo_estimado_usd}")
print()
print(json.dumps(parecer_v1.model_dump(), indent=2, ensure_ascii=False) if parecer_v1 else f"FALHOU: {tel_v1.erro}")

## 8. Prompt v2 — papel, glossário e fronteira explícita

As duas versões não diferem em "seja mais detalhado". Diferem em **três decisões de design**:

1. **Papel e enquadramento regulatório** — analista de PLD, não "assistente que analisa dados".
   Muda o vocabulário de saída de genérico para técnico.
2. **Glossário de tipologias** — fracionamento/*smurfing*, uso intensivo de espécie, remessa
   internacional atípica, interposição de pessoa. Sem isso o modelo inventa nome de tipologia,
   e nome inventado não casa com taxonomia de compliance nenhuma.
3. **Fronteira explícita** — instrução direta de que os números já foram calculados e **não devem
   ser recalculados nem estimados**, e de que cada *red flag* precisa citar um dado presente no
   dossiê. É a defesa contra alucinação numérica, que é o risco concreto aqui: um parecer com número
   errado é pior que nenhum parecer, porque parece confiável.

In [ ]:
SYSTEM_V2 = """Voce e analista de Prevencao a Lavagem de Dinheiro (PLD) de uma instituicao financeira brasileira, na mesa de triagem.

Sua funcao e INTERPRETAR indicadores ja calculados e redigir parecer tecnico para o analista humano que vai decidir sobre o caso. Voce nao calcula: todos os numeros do dossie foram apurados por rotina deterministica auditada.

REGRAS INEGOCIAVEIS:
- Nao recalcule, nao estime e nao infira nenhum numero que nao esteja no dossie.
- Toda red flag deve citar um dado presente no dossie. Sem dado, sem flag.
- Se os indicadores nao sustentarem suspeita, atribua risco baixo. Falso positivo consome hora de analista e desgasta o cliente.

TIPOLOGIAS DE REFERENCIA (use a nomenclatura, ou 'Nao caracterizada'):
- Fracionamento (smurfing): divisao de valor em operacoes menores para evitar limite de reporte
- Uso intensivo de especie: concentracao em deposito/saque em dinheiro
- Remessa internacional atipica: operacao em moeda estrangeira sem lastro compativel com o perfil
- Interposicao de pessoa: conta usada como passagem, entrada e saida rapidas
- Incompatibilidade com o perfil: volume desalinhado do historico do proprio cliente

NIVEIS: baixo (compativel com o perfil) | medio (merece observacao, sem elemento conclusivo) | alto (elemento objetivo que justifica analise humana prioritaria)

Responda EXCLUSIVAMENTE com um objeto JSON valido, sem texto antes ou depois."""

USER_V2 = f"""DOSSIE DO CLIENTE (valores em BRL, ja convertidos e tratados)

{json.dumps(dossie, ensure_ascii=False, indent=2)}

CONTEXTO DAS REGRAS QUE ANALISARAM ESTE CLIENTE
- Regra de fracionamento: 3+ operacoes no mesmo dia somando mais de R$ 50.000, com nenhuma atingindo R$ 20.000.
- Regra de valor atipico: operacao acima de 5x a mediana do proprio cliente, aplicada a clientes com 4+ operacoes.
O campo 'regras_acionadas' ja informa o resultado dessas regras. Nao reavalie os limites.

TAREFA
Produza o parecer no formato:
{{
  "nivel_risco": "baixo" | "medio" | "alto",
  "tipologia_suspeita": "<nome da tipologia ou 'Nao caracterizada'>",
  "red_flags": ["<indicio ancorado em dado do dossie>", "..."],
  "justificativa": "<3 a 5 frases: o que os indicadores mostram, o que ainda falta para concluir, e o que o analista humano deveria verificar>"
}}"""

parecer_v2, tel_v2 = llm.chamar(SYSTEM_V2, USER_V2)

print(f"status={tel_v2.status} | tentativas={tel_v2.tentativas} | cacheado={tel_v2.cacheado}")
print(f"tokens: {tel_v2.tokens_entrada} entrada + {tel_v2.tokens_saida} saida")
print(f"latencia: {tel_v2.latencia_s}s | custo estimado: US$ {tel_v2.custo_estimado_usd}")
print()
print(json.dumps(parecer_v2.model_dump(), indent=2, ensure_ascii=False) if parecer_v2 else f"FALHOU: {tel_v2.erro}")

## 9. Comparação das duas versões

⚠️ **Escreva sua leitura aqui DEPOIS de rodar as duas células.** Compare em quatro eixos:

1. **Nomenclatura da tipologia** — a v2 devolveu um nome do glossário ou uma descrição genérica?
2. **Ancoragem das red flags** — cada uma cita número do dossiê, ou tem alguma inventada?
3. **Alucinação numérica** — apareceu algum valor que não está no dossiê? *Se apareceu na v1, isso é
   o achado mais forte do notebook inteiro*: é a demonstração empírica de por que cálculo fica em
   pandas. Cite o número errado explicitamente.
4. **Custo** — a v2 gasta mais tokens de entrada. Quanto a mais, e valeu a pena?

In [ ]:
comparacao = pd.DataFrame([
    {"versao": "v1 — mínima",
     "nivel_risco": parecer_v1.nivel_risco if parecer_v1 else "—",
     "tipologia": parecer_v1.tipologia_suspeita if parecer_v1 else "—",
     "n_red_flags": len(parecer_v1.red_flags) if parecer_v1 else 0,
     "tokens_entrada": tel_v1.tokens_entrada, "tokens_saida": tel_v1.tokens_saida,
     "latencia_s": tel_v1.latencia_s, "custo_usd": tel_v1.custo_estimado_usd,
     "tentativas": tel_v1.tentativas, "status": tel_v1.status},
    {"versao": "v2 — papel + glossário",
     "nivel_risco": parecer_v2.nivel_risco if parecer_v2 else "—",
     "tipologia": parecer_v2.tipologia_suspeita if parecer_v2 else "—",
     "n_red_flags": len(parecer_v2.red_flags) if parecer_v2 else 0,
     "tokens_entrada": tel_v2.tokens_entrada, "tokens_saida": tel_v2.tokens_saida,
     "latencia_s": tel_v2.latencia_s, "custo_usd": tel_v2.custo_estimado_usd,
     "tentativas": tel_v2.tentativas, "status": tel_v2.status},
]).set_index("versao")

display(comparacao)

for rot, par in [("v1", parecer_v1), ("v2", parecer_v2)]:
    if par:
        print(f"\n{'='*70}\n{rot} — red flags")
        for f in par.red_flags:
            print(f"  • {f}")
        print(f"\n{rot} — justificativa\n  {par.justificativa}")

## 10. Tratamento de resposta malformada

O enunciado pede tratamento de resposta malformada; abaixo eu **demonstro** que a camada funciona,
em vez de só afirmar. São sete formas de resposta quebrada aplicadas direto ao validador — sem
gastar quota da API e com resultado reproduzível.

A defesa tem três camadas:

1. **`response_format={"type": "json_object"}`** — força JSON sintaticamente válido no lado do
   provedor. Não garante que os *campos* estejam certos.
2. **Validação Pydantic** — pega campo faltando, tipo errado, lista vazia e valor fora do
   vocabulário permitido.
3. **Retry realimentado** — o erro de validação volta ao modelo como mensagem. Corrigir com o erro
   em mãos funciona muito melhor que repetir a mesma pergunta.

Falhando as três, a função devolve `None` com `status="falha_validacao"` e **não levanta exceção** —
no lote do Nível 2, um cliente problemático não pode derrubar os outros nove.

In [ ]:
casos = [
    ("bem formado",     '{"nivel_risco":"alto","tipologia_suspeita":"Fracionamento","red_flags":["3 ops no mesmo dia"],"justificativa":"Padrao compativel com estruturacao de valores para evitar limite de reporte."}'),
    ("cerca markdown",  '```json\n{"nivel_risco":"medio","tipologia_suspeita":"Valor atipico","red_flags":["operacao 12x a mediana"],"justificativa":"Operacao muito acima do padrao historico observado no cliente."}\n```'),
    ("preambulo",       'Claro! Aqui esta a analise:\n{"nivel_risco":"baixo","tipologia_suspeita":"Nao caracterizada","red_flags":["sem indicios"],"justificativa":"Movimentacao compativel com o perfil declarado do cliente."}'),
    ("vocab invalido",  '{"nivel_risco":"altissimo","tipologia_suspeita":"X","red_flags":["y"],"justificativa":"texto suficientemente longo para passar"}'),
    ("campo faltando",  '{"nivel_risco":"alto","red_flags":["y"],"justificativa":"texto suficientemente longo para passar"}'),
    ("red_flags vazia", '{"nivel_risco":"alto","tipologia_suspeita":"X","red_flags":[],"justificativa":"texto suficientemente longo para passar"}'),
    ("json quebrado",   '{"nivel_risco": "alto", red_flags: }'),
]

linhas = []
for nome, texto in casos:
    obj, erro = llm.validar(texto, llm.ParecerPLD)
    linhas.append({
        "caso": nome,
        "resultado": "aceito" if obj else "rejeitado",
        "motivo": "—" if obj else erro.split("[")[0].strip().rstrip(":"),
        "tipo_do_erro": "—" if obj else (erro.split("'type': '")[1].split("'")[0] if "'type': '" in erro else "—"),
    })

pd.DataFrame(linhas).set_index("caso")

Os três primeiros são respostas **utilizáveis** que chegam sujas — a camada de extração as recupera
sem gastar retry. Os quatro últimos são respostas **inválidas**, e cada uma é barrada por um motivo
distinto e identificável, o que permite decidir se vale retry ou se o caso vai para exceção.

O caso `vocab invalido` é o que mais importa: `"altissimo"` é JSON perfeitamente válido e uma
resposta plausível em português. Sem o `Literal` no schema, ele passaria — e quebraria a comparação
automática do Nível 2 silenciosamente, meses depois, sem nenhum erro de execução.

## 11. Conclusões

⚠️ **Escreva esta seção com suas palavras depois de rodar tudo.** Três parágrafos, e nenhum deles
sobre o que você programou:

**Sobre os dados.** Três defeitos de qualidade em 20 operações, e dois deles alteram o resultado da
triagem em direções opostas — a duplicata cria alerta falso, a moeda não convertida esconde alerta
verdadeiro. Nenhum apareceria como erro de execução.

**Sobre os clientes.** O que você concluiu de PLD: CLI-A-1 apresenta padrão de fracionamento;
CLI-A-4 tem uma remessa internacional que destoa em uma ordem de grandeza do resto da conta;
CLI-A-2 tem volume alto sem característica de fracionamento e a regra corretamente não o sinaliza.

**Sobre a divisão de trabalho.** O que você aprendeu sobre onde termina a regra e onde começa o
modelo — e, se apareceu alucinação numérica na v1, use isso como evidência.

---

**Próximo nível:** as mesmas funções de `pipeline.py` rodam sobre 322 operações e 30 clientes sem
alteração, e as ferramentas do agente reaproveitam `dossie_cliente()`.